# ML-08 — Capstone Modeling Lane
**Lane:** Refresh / Content Opportunity Scoring
**Goal:** Fit the right model for this lane and compare it to the Week-4 baseline on the *same* data split and the *same* metric.

Sections (in order): 1) Method choice and why → 2) Split design → 3) Train + compare vs my baseline → 4) Errors and interpretation → 5) Self-check.

In [ ]:
# --- Setup (clone + cd, same pattern used in prior weeks) ---
import os, sys, subprocess

REPO_URL = "https://github.com/ramanchauhan2271-dev/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone", REPO_URL], check=True)
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, average_precision_score,
    precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.inspection import permutation_importance

pd.set_option("display.max_columns", 60)
np.random.seed(42)

## 1) Method choice and why

**Task type:** binary classification — `is_declining_label = (trend_direction == "down")` (a page is a refresh candidate if it's trending down).

**Candidates from this week's menu:**
- *Logistic Regression* — fast, interpretable coefficients, good sanity-check model, but assumes linear decision boundary in feature space.
- *Decision Tree* — interpretable rules, but a single tree overfits easily on ~30k rows with 50+ engineered features.
- *Random Forest* — handles non-linear interactions between features (e.g. traffic trend × backlink decay × content age) without heavy tuning, robust to outliers, gives permutation importance for free.
- *Gradient Boosting* — usually the strongest performer on tabular data like this, but more prone to overfitting and slower to validate carefully — "where safe" per the assignment, meaning: only if the grouped validation shows it generalizes.

**My choice:** train Logistic Regression as an interpretable floor, Random Forest as the main workhorse, and Gradient Boosting as a stretch model — then let the Week-4-baseline comparison (same split, same metric) decide which one actually earns its complexity. The assignment explicitly says "does not reward complexity alone," so all three get evaluated on identical footing and I pick the winner in Section 3, not here.

**Metric:** the lane is a *ranking/triage* problem (which pages to refresh first), not a balanced classification problem — refresh candidates are a minority class. So the primary metric is **PR-AUC (average precision)**, with ROC-AUC and precision/recall @ top-K reported alongside for context. This matches the baseline's own scoring logic (a ranked queue), so it's a fair apples-to-apples comparison.

## 2) Split design

**Leakage risk:** rows in `refresh_feature_vector.csv` are individual pages, but many pages share the same `client_id` / `domain`. If I split by row, pages from the same site end up in both train and test, and the model can "memorize" site-level patterns (e.g. a domain's typical traffic trend) instead of learning generalizable content-decline signal. That would inflate my score relative to the baseline unfairly.

**Design:** a **grouped split by `client_id`** (fallback to `domain` if `client_id` isn't present) using `GroupShuffleSplit`, so an entire site's pages land entirely in train or entirely in test. This is the same split I'll reuse for the baseline comparison, so both are scored on the exact same held-out clients.

In [ ]:
# --- Load features + baseline (outputs from earlier weeks' pipeline stages) ---
FEATURES_PATH = "data/processed/refresh_feature_vector.csv"
BASELINE_PATH = "data/processed/baseline_refresh_queue.csv"

features = pd.read_csv(FEATURES_PATH)
baseline = pd.read_csv(BASELINE_PATH)

print("features:", features.shape)
print("baseline:", baseline.shape)
features.head()

In [ ]:
# --- Target + grouping column ---
TARGET = "is_declining_label"
GROUP_COL = "client_id" if "client_id" in features.columns else "domain"

assert TARGET in features.columns, f"{TARGET} not found — check 01_prepare_features.py output"
print("Group column:", GROUP_COL, "| unique groups:", features[GROUP_COL].nunique())
print("Positive rate:", features[TARGET].mean().round(3))

In [ ]:
# --- Feature matrix: drop identifiers/leakage columns, keep engineered numeric/categorical features ---
DROP_COLS = [TARGET, GROUP_COL, "page_id", "url", "trend_direction"]
DROP_COLS = [c for c in DROP_COLS if c in features.columns]

X = features.drop(columns=DROP_COLS)
y = features[TARGET].astype(int)
groups = features[GROUP_COL]

# one-hot encode any remaining categoricals
X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median(numeric_only=True))

print("X shape:", X.shape)

In [ ]:
# --- Grouped 80/20 split, same split object reused for baseline scoring below ---
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

train_groups = set(groups.iloc[train_idx])
test_groups = set(groups.iloc[test_idx])
print("Leakage check — overlapping groups:", len(train_groups & test_groups))  # must be 0
print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Test positive rate:", y_test.mean().round(3))

## 3) Train + compare vs my baseline

Train all three candidate models on the *same* `X_train`, score on the *same* `X_test`, and pull the baseline's score for the identical test rows so the comparison is apples-to-apples.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=10,
        class_weight="balanced", random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
    ),
}

results = {}
proba_store = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    proba_store[name] = proba
    results[name] = {
        "PR-AUC": average_precision_score(y_test, proba),
        "ROC-AUC": roc_auc_score(y_test, proba),
        "Precision@ 0.5": precision_score(y_test, proba > 0.5, zero_division=0),
        "Recall@ 0.5": recall_score(y_test, proba > 0.5, zero_division=0),
        "F1@ 0.5": f1_score(y_test, proba > 0.5, zero_division=0),
    }

results_df = pd.DataFrame(results).T
results_df

In [ ]:
# --- Pull baseline's score for the SAME held-out test rows ---
# baseline_refresh_queue.csv is keyed by page_id and carries the hand-rule's numeric score
key = "page_id" if "page_id" in features.columns else features.columns[0]

test_keys = features.iloc[test_idx][key]
baseline_test = baseline.set_index(key).reindex(test_keys)["baseline_score"]  # adjust col name if different

baseline_pr_auc = average_precision_score(y_test.values, baseline_test.values)
baseline_roc_auc = roc_auc_score(y_test.values, baseline_test.values)

print(f"Baseline PR-AUC:  {baseline_pr_auc:.3f}")
print(f"Baseline ROC-AUC: {baseline_roc_auc:.3f}")

In [ ]:
# --- Final model-vs-baseline table (the deliverable the assignment asks for) ---
comparison = results_df[["PR-AUC", "ROC-AUC"]].copy()
comparison.loc["Baseline (Week 4)"] = [baseline_pr_auc, baseline_roc_auc]
comparison["Lift vs baseline (PR-AUC)"] = (comparison["PR-AUC"] - baseline_pr_auc).round(3)
comparison = comparison.sort_values("PR-AUC", ascending=False)
comparison

In [ ]:
best_model_name = comparison.drop("Baseline (Week 4)").index[0]
print(f"Best model: {best_model_name}")
print(comparison.loc[[best_model_name, 'Baseline (Week 4)']])

## 4) Errors and interpretation

Look at *where* the winning model disagrees with the ground truth, and which features drive its decisions (permutation importance, since it works for any of the three model types and isn't biased toward high-cardinality features the way built-in feature_importances_ can be).

In [ ]:
best_model = models[best_model_name]
best_proba = proba_store[best_model_name]
pred = (best_proba > 0.5).astype(int)

cm = confusion_matrix(y_test, pred)
print("Confusion matrix [rows=actual, cols=predicted]:")
print(pd.DataFrame(cm, index=["Actual: stable", "Actual: declining"],
                    columns=["Pred: stable", "Pred: declining"]))

In [ ]:
# --- Permutation importance on held-out data ---
perm = permutation_importance(
    best_model, X_test, y_test, n_repeats=10, random_state=42, scoring="average_precision", n_jobs=-1
)
importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df.head(15)

In [ ]:
importance_df.head(15).plot.barh(x="feature", y="importance_mean", xerr="importance_std", figsize=(8,6))
plt.gca().invert_yaxis()
plt.title(f"Permutation importance — {best_model_name}")
plt.tight_layout()
plt.show()

In [ ]:
# --- Inspect a few of the worst false negatives and false positives ---
test_view = features.iloc[test_idx].copy()
test_view["y_true"] = y_test.values
test_view["y_proba"] = best_proba

false_negatives = test_view[(test_view["y_true"] == 1) & (test_view["y_proba"] < 0.3)]
false_positives = test_view[(test_view["y_true"] == 0) & (test_view["y_proba"] > 0.7)]

print("Worst false negatives (missed real decliners):")
display_cols = [c for c in [key, "domain", "y_true", "y_proba"] if c in test_view.columns]
print(false_negatives[display_cols].head())

print("\nWorst false positives (flagged stable pages as declining):")
print(false_positives[display_cols].head())

**What the errors look like (fill in after running on real data):**
- *False negatives* — pages the model rated low-risk but were actually declining. Check whether these share a common blind spot (e.g. seasonal content where the "decline" is cyclical, not structural — a feature the model doesn't see).
- *False positives* — pages flagged as declining that were actually stable. Check if these are recently published pages still in a normal ramp-up dip, which the model may be confusing with genuine decline.
- Compare this error pattern to the baseline's errors: does the model fix the baseline's blind spots, or just shift where the mistakes happen?

## 5) Self-check

- [ ] Model compared against the baseline on the **same test split** and the **same metric** (PR-AUC) — done via `GroupShuffleSplit` reused for both.
- [ ] Split is leak-safe — grouped by `client_id`/`domain`, verified zero group overlap between train/test above.
- [ ] Method choice explained *before* seeing results (Section 1), not reverse-engineered after.
- [ ] Reported more than one metric (PR-AUC, ROC-AUC, precision/recall) so a single number isn't hiding a weakness.
- [ ] Interpreted errors, not just metrics — looked at actual false positive/negative rows, not only the confusion matrix.
- [ ] Complexity was earned — Gradient Boosting only kept if it actually beats Random Forest/Logistic Regression on held-out data, not by default.
- [ ] AI-drafted code was read and validated line by line before treating results as final — re-check column names (`baseline_score`, `page_id`, `client_id`) against what actually exists in *your* `refresh_feature_vector.csv` / `baseline_refresh_queue.csv`, since these may differ slightly from the starter schema.